# ⚖️ Veredito B3+B4a — país/política e o resto do planeta deixam a crônica mais viva?

Notebook de veredito da **Fase 3 do Degrau 1** (B3 País e política: estabilidade S com reversão à média ancorada em WGI/ICRG, câmbio-índice, royalties sobre o opex como sink, motor de eventos endógeno com frequências latino-americanas 2015–2025 e o acoplamento preço→nacionalismo; B4a Importações: pseudo-participante elástico na paridade p*·(1+tarifa+frete), com a conta "externo" auditável no ledger). Protocolo **pré-registrado**:

- Corridas **pareadas**: seeds 42, 7, 11, 23 × (B1 ligado, B3+B4a desligado) × (B1 ligado, B3+B4a ligado), 8 empresas, 120 ticks — só a flag `country_v2` muda. A base já inclui o mercado vivo (B1): esta fase é julgada sobre ele, porque o acoplamento B1→B3 lê a âncora pública do B1.
- Métricas: eventos salientes da crônica por corrida (sem `flows`/`founding`/`end`), eventos políticos por tipo (`politics` com `data.event`), amplitude de preços (mediana de max/min por commodity nos frames), falências e suas causas, royalties pagos (sink `spend_royalty` nos flows), comércio externo (massa importada/exportada e a conta externo em US$).
- **Regra de aprovação (régua do PRD):** (1) ledger fecha em TODAS as corridas — dinheiro incluindo a conta externo (`Σcaixa = inicial + money_in + money_from_external − money_out − money_to_external`) e massa incluindo a fonte `imported`; **E** (2) eventos salientes ON > OFF em ≥ 3 de 4 seeds; **E** (3) ≥ 6 eventos políticos por corrida ON em ≥ 3 de 4 seeds; **E** (4) ≥ 4 empresas vivas ao fim de cada corrida ON; **E** (5) comércio externo (importado + exportado) > 0 em ≥ 3 de 4 seeds ON. Fora isso: REPROVADO/AJUSTAR.

Regra do fundador: simulação de verdade roda AQUI, no Kaggle — nunca na VM de desenvolvimento.

In [ ]:
# 1) Fonte do pacote: dataset privado do bundle mais recente
#    (mundo-autonomo-srcN — criado a partir do repo com
#    scripts/kaggle_push_kernel.py --bundle-src no lançamento).
#    O pacote e puro Python: NAO precisa de pip install. Basta src/ no path.
#    LOADER RECURSIVO (26/10/2026): o Kaggle mudou o layout do mount de
#    datasets — agora eles aparecem em /kaggle/input/datasets/<dono>/<slug>/
#    com a arvore JA EXTRAIDA (sem o zip dentro), e o caminho classico
#    /kaggle/input/<slug>/ nao existe mais. A busca aceita os DOIS layouts:
#    qualquer 'src/mundo/__init__.py' sob /kaggle/input, com fallback
#    para o zip mundo-src.zip se ele existir.
import glob, os, sys, zipfile

inits = sorted(glob.glob('/kaggle/input/**/src/mundo/__init__.py',
                         recursive=True))
zip_cands = sorted(glob.glob('/kaggle/input/**/mundo*.zip', recursive=True))
if inits:
    src_root = os.path.dirname(os.path.dirname(os.path.dirname(inits[-1])))
elif zip_cands:
    zipfile.ZipFile(zip_cands[-1]).extractall('/kaggle/working/pkg')
    src_root = '/kaggle/working/pkg'
else:
    raise SystemExit('Fonte do pacote nao encontrada (loader recursivo): '
                     'anexe o dataset pascalcoimbra/mundo-autonomo-srcN '
                     'a este kernel.')
print('pacote em', src_root)
sys.path.insert(0, os.path.join(src_root, 'src'))
import mundo
print('mundo', mundo.__version__, 'importado de', src_root)


## As corridas pareadas
Cada corrida imprime o placar completo no objeto de resultados; a conservação é verificada corrida a corrida a partir dos contadores do motor — dinheiro **incluindo a conta externo** do B4a e massa **incluindo a fonte `imported`**, que é o invariante novo desta fase.

In [ ]:
# 2) Corridas pareadas: MESMAS seeds, base B1 x B1+B3/B4a.
import statistics

from mundo import chain
from mundo.engine import Engine

SEEDS = [42, 7, 11, 23]
TICKS = 120
SALIENT_IGNORED = {"flows", "founding", "end"}


def metrics(eng, res):
    kinds = {}
    politics = {}
    royalty = 0.0
    export_value = 0.0
    for e in res["chronicle"]:
        kinds[e["kind"]] = kinds.get(e["kind"], 0) + 1
        if e["kind"] == "politics":
            t = e["data"]["event"]
            politics[t] = politics.get(t, 0) + 1
        if e["kind"] == "flows":
            for rec in e["data"]["companies"].values():
                royalty += -rec["flows"].get("spend_royalty", 0.0)
                export_value += rec["flows"].get("export_sale", 0.0)
    salient = sum(v for k, v in kinds.items() if k not in SALIENT_IGNORED)
    amps = []
    for c in chain.COMMODITIES:
        series = [f["prices"][c] for f in res["frames"]]
        amps.append(max(series) / min(series))
    total = sum(c.cash for c in eng.companies)
    money_ok = abs(total - (eng.initial_cash + eng.money_in
                            + eng.money_from_external - eng.money_out
                            - eng.money_to_external)) < 1e-4
    mass_ok = True
    for c in chain.COMMODITIES:
        led = eng.ledger_mass[c]
        stocks = sum(comp.stocks[c] for comp in eng.companies)
        bal = led["extracted"] + led["produced"] \
            + led.get("imported", 0.0) - led["consumed"] \
            - led["sold_external"]
        if abs(stocks - bal) > 1e-3:
            mass_ok = False
    causes = {}
    for e in res["chronicle"]:
        if e["kind"] == "bankruptcy":
            causes[e["data"]["cause"]] = causes.get(e["data"]["cause"], 0) + 1
    top = res["scoreboard"][0]
    return {
        "salient": salient,
        "rivalry": kinds.get("rivalry", 0),
        "politics_total": sum(politics.values()),
        "politics_by_type": politics,
        "bankruptcies": kinds.get("bankruptcy", 0),
        "bankruptcy_causes": causes,
        "royalty_paid": royalty,
        "export_value": export_value,
        "imported_mass": sum(eng.ledger_mass[c].get("imported", 0.0)
                             for c in chain.COMMODITIES),
        "money_to_external": eng.money_to_external,
        "money_from_external": eng.money_from_external,
        "price_amplitude_median": statistics.median(amps),
        "ledger_ok": money_ok and mass_ok,
        "top_cash_name": top["name"], "top_cash": top["cash"],
        "max_stage": max(r["stage"] for r in res["scoreboard"]),
        "alive": sum(1 for r in res["scoreboard"] if r["alive"]),
        "supercomputers": int(eng.total_supercomputers),
    }


results = {}
for seed in SEEDS:
    for label, on in (("off", False), ("on", True)):
        eng = Engine(seed=seed, n_companies=8, market_v2=True,
                     country_v2=on)
        res = eng.run(TICKS)
        results[(seed, label)] = metrics(eng, res)
        print(f"seed {seed} B3+B4a {label}: pronto")


## O veredito
A tabela compara OFF × ON por seed e a última linha imprime o veredito pela regra pré-registrada. Os eventos `politics` só existem com o B3 ligado — eles são parte do ganho de vivacidade, mas **não** contam sozinhos: a régua exige também mais eventos salientes no total, ledger fechando com a conta externa e o participante externo realmente operando (importando/exportando de verdade).

In [ ]:
# 3) Comparativo e VEREDITO (regra pre-registrada no topo do notebook)
import json

print(f"{'seed':>5} {'metrica':<26} {'B3 off':>10} {'B3 on':>10}")
wins_salient = 0
politics_ok = 0
trade_ok = 0
ledger_all = True
alive_ok = True
amps_off, amps_on = [], []
for seed in SEEDS:
    off, on = results[(seed, "off")], results[(seed, "on")]
    ledger_all = ledger_all and off["ledger_ok"] and on["ledger_ok"]
    alive_ok = alive_ok and on["alive"] >= 4
    if on["salient"] > off["salient"]:
        wins_salient += 1
    if on["politics_total"] >= 6:
        politics_ok += 1
    if on["imported_mass"] > 0 or on["money_from_external"] > 0:
        trade_ok += 1
    amps_off.append(off["price_amplitude_median"])
    amps_on.append(on["price_amplitude_median"])
    for key in ("salient", "rivalry", "politics_total", "bankruptcies"):
        print(f"{seed:>5} {key:<26} {off[key]:>10.0f} {on[key]:>10.0f}")
    print(f"{seed:>5} {'price_amplitude_median':<26} "
          f"{off['price_amplitude_median']:>10.2f} "
          f"{on['price_amplitude_median']:>10.2f}")
    print(f"{seed:>5} {'royalties pagos (so on)':<26} {'-':>10} "
          f"{on['royalty_paid']:>10.0f}")
    print(f"{seed:>5} {'importado t (so on)':<26} {'-':>10} "
          f"{on['imported_mass']:>10.1f}")
    print(f"{seed:>5} {'politica por tipo (on)':<26} "
          f"{str(on['politics_by_type'])[:44]:>54}")
    print(f"{seed:>5} {'causas de falencia (on)':<26} "
          f"{str(on['bankruptcy_causes'])[:44]:>54}")
    print(f"{seed:>5} {'placar: lider de caixa':<26} "
          f"{off['top_cash_name'][:10]:>10} {on['top_cash_name'][:10]:>10}")
print()
print(f"amplitude mediana de precos: off "
      f"{statistics.median(amps_off):.2f}x | on "
      f"{statistics.median(amps_on):.2f}x")
print(f"eventos salientes: B3+B4a on vence em "
      f"{wins_salient}/{len(SEEDS)} seeds")
print(f"corridas on com >= 6 eventos politicos: {politics_ok}/{len(SEEDS)}")
print(f"corridas on com comercio externo ativo: {trade_ok}/{len(SEEDS)}")
print(f"ledger fecha (dinheiro c/ externo + massa c/ imported) "
      f"em todas: {ledger_all}")
print(f"corridas on com >= 4 empresas vivas: {alive_ok}")

approved = (ledger_all and alive_ok and wins_salient >= 3
            and politics_ok >= 3 and trade_ok >= 3)
print()
print("VEREDITO B3+B4a:", "APROVADO — a cronica fica visivelmente mais "
      "viva com pais, politica e o resto do planeta, e os ledgers "
      "fecham incluindo a conta externa." if approved else
      "REPROVADO/AJUSTAR — ver metricas acima: a regua do PRD nao foi "
      "atingida; calibrar parametros antes de qualquer nova tentativa.")
with open("/kaggle/working/veredito-b3.json", "w", encoding="utf-8") as f:
    json.dump({f"{s}-{l}": m for (s, l), m in results.items()}, f,
              ensure_ascii=False, indent=1)
print("detalhes em /kaggle/working/veredito-b3.json")
